# Lesson 5 — Sanity-check the VAD dataset

We just built `vad_dataset.py`. Before we train on it for hours, we need to verify the data looks right. Garbage in, garbage out — debugging a model trained on broken data takes 10× longer than checking the data once up front.

This notebook does three checks:

1. **Single sample** — pull one example, visualize the spectrogram with labels overlaid. Do the labels track the speech regions?
2. **Multiple samples** — pull ten random examples. Are SNRs varied? Are noise types varied? Are some clean?
3. **DataLoader benchmark** — how fast can we generate batches? On an L40S, the GPU will starve if the data pipeline is too slow.

Run cells top to bottom. Look at every plot.


## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '/workspace')  # so `import vad_dataset` works; adjust if needed

import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import librosa.display
import warnings
warnings.filterwarnings('ignore')

from vad_dataset import VADDataset

plt.rcParams['figure.dpi'] = 100


## 1. Build the dataset

In [ ]:
ds = VADDataset(
    speech_dir="/workspace/data/LibriSpeech/train-clean-100",
    noise_dir="/workspace/data/musan/noise",
    music_dir="/workspace/data/musan/music",
    clip_seconds=4.0,
    sample_rate=16000,
    snr_range_db=(-5, 25),
    seed=0,
)
print(f"Dataset advertises length: {len(ds)} virtual samples")


## 2. Visualize a single sample

Pull example 0 and plot the log-mel spectrogram. Overlay the frame-level labels as a black bar at the bottom: tall when the model should predict speech, flat when not.

**Check**: do the dark patches (silence) line up with the flat label regions, and the bright patches (speech) line up with the tall label regions?


In [ ]:
feat, lab = ds[0]
print(f"Features: {feat.shape}, dtype {feat.dtype}, range [{feat.min():.2f}, {feat.max():.2f}]")
print(f"Labels:   {lab.shape}, dtype {lab.dtype}, speech fraction: {lab.float().mean():.2%}")

fig, ax = plt.subplots(figsize=(14, 5))
librosa.display.specshow(
    feat.numpy(),
    sr=16000, hop_length=160, x_axis='time', y_axis='mel',
    cmap='magma', ax=ax,
)
ax.set_title("Log-mel spectrogram of one VAD training sample")

# Overlay labels as a black bar in the bottom 10% of the plot
ax_lab = ax.twinx()
times = np.arange(len(lab)) * 160 / 16000
ax_lab.fill_between(times, 0, lab.numpy(), step='mid', alpha=0.4, color='cyan')
ax_lab.set_ylim(0, 8)  # squash the bar to the bottom
ax_lab.set_yticks([])

plt.tight_layout()
plt.show()


## 3. Pull ten samples and look at variety

We want to confirm: some clips are noisy, some clean, some have music, some have nothing.


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 14))
for i, ax in enumerate(axes.flat):
    feat, lab = ds[i + 1]
    img = librosa.display.specshow(
        feat.numpy(), sr=16000, hop_length=160,
        x_axis='time', y_axis='mel', cmap='magma', ax=ax,
    )
    ax2 = ax.twinx()
    times = np.arange(len(lab)) * 160 / 16000
    ax2.fill_between(times, 0, lab.numpy(), step='mid', alpha=0.4, color='cyan')
    ax2.set_ylim(0, 8); ax2.set_yticks([])
    ax.set_title(f"Sample {i+1}: {lab.float().mean():.0%} speech")
plt.tight_layout()
plt.show()


**What to look for:**

- Some samples should show clean speech (sharp spectra, clear formants).
- Others should show noisy speech (cloudier spectra, but labels should still mark the speech regions correctly).
- A few might be very low-SNR (heavily noised). The labels still track the underlying clean speech.
- Speech fraction varies clip to clip. Anywhere from 30% to 90% is normal depending on how much pause was in the source LibriSpeech file.

If you see labels that obviously don't track the bright bands, or all samples look identical, something is wrong — ping me.

---

## 4. Speed test — can we feed the GPU?

The L40S can chew through batches very fast. If our data loading is slower than the GPU can consume, training is data-bound and we waste the GPU. Let's measure.


In [ ]:
# Single-worker baseline
loader = DataLoader(ds, batch_size=32, num_workers=0, shuffle=False)
t0 = time.time()
for i, (feat, lab) in enumerate(loader):
    if i >= 10:
        break
dt = time.time() - t0
print(f"num_workers=0:   {10 * 32 / dt:.1f} samples/sec  ({dt:.2f}s for 320 samples)")


In [ ]:
# Multi-worker — this is what real training will use
loader = DataLoader(ds, batch_size=32, num_workers=8, shuffle=False, persistent_workers=True)
# Warm up the workers
for _ in loader:
    break
t0 = time.time()
for i, (feat, lab) in enumerate(loader):
    if i >= 20:
        break
dt = time.time() - t0
print(f"num_workers=8:   {20 * 32 / dt:.1f} samples/sec  ({dt:.2f}s for 640 samples)")


**Rule of thumb for L40S**:

- A small VAD model trains at roughly 5,000–15,000 samples/sec on the L40S.
- The data loader needs to match or exceed that, otherwise we're starving the GPU.
- With `num_workers=8` on a modern CPU you should be in the 1000–4000 samples/sec range — workable but possibly the bottleneck. If it's too slow, we'll cache mel features to disk in a later optimization. For now: don't worry, just note the number.

---

## 5. Sanity-check the label distribution

A common silent bug: if your threshold or pipeline is off, you might end up with labels that are all-speech or all-silence. Let's confirm we have a healthy mix.


In [ ]:
speech_fractions = []
for i in range(100):
    _, lab = ds[i]
    speech_fractions.append(lab.float().mean().item())

speech_fractions = np.array(speech_fractions)
print(f"Speech fraction over 100 samples: mean={speech_fractions.mean():.2%}, "
      f"min={speech_fractions.min():.2%}, max={speech_fractions.max():.2%}")

plt.figure(figsize=(10, 3))
plt.hist(speech_fractions, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel("Fraction of frames labeled 'speech'")
plt.ylabel("# samples")
plt.title("Speech fraction across 100 training samples")
plt.tight_layout()
plt.show()


**What to look for:**

- Mean speech fraction around 60-80% — LibriSpeech is mostly continuous reading with short pauses.
- A few samples below 50% (clips that happened to start during long pauses).
- Nothing at 0% (would mean the energy threshold is too high — no frames pass) or 100% (threshold too low — every frame passes). Both of those would be bugs.

If the histogram is sharply skewed to one extreme, the threshold needs adjusting. Default `-40 dB` works well for LibriSpeech.

---

## 6. (Optional) Listen to a sample

The most useful sanity check is your ears. Save one mixed clip and listen — does the speech still sound recognizable through the noise at low SNR? If you can't tell what's being said, your model probably can't either.


In [ ]:
import soundfile as sf

# Re-run the pipeline manually to get the raw audio (the dataset doesn't return it)
# This is just for inspection — listen to the mix.
from vad_dataset import load_audio_mono, fit_length, mix_at_snr
import random as _r

rng = _r.Random(42)
speech = load_audio_mono(rng.choice(ds.speech_files), 16000)
speech = fit_length(speech, ds.clip_len, rng)
noise  = load_audio_mono(rng.choice(ds.noise_files), 16000)
noise  = fit_length(noise, ds.clip_len, rng)

for snr in [20, 5, -5]:
    mix = mix_at_snr(speech, noise, snr)
    sf.write(f"/tmp/sample_snr{snr}.wav", mix, 16000)
    print(f"Wrote /tmp/sample_snr{snr}.wav at {snr} dB SNR")

# In Jupyter, you can play these inline:
from IPython.display import Audio, display
for snr in [20, 5, -5]:
    print(f"\n--- SNR {snr} dB ---")
    display(Audio(f"/tmp/sample_snr{snr}.wav"))


**This is the most important cell.** Listen to all three.

- At **+20 dB**: speech is dominant. Noise is faintly audible in the background.
- At **+5 dB**: noise is clearly there. Speech is still understandable but you have to focus.
- At **-5 dB**: noise dominates. Speech might be barely intelligible — and this is exactly the regime where a great VAD shines and a bad VAD fails.

Your training data spans this full range. The model has to learn to mark speech regions correctly across all of it.

---

## You're done with lesson 5 when…

- All five checks above passed (single sample, ten samples, speed test, label histogram, listening test).
- You're confident the data going into training is correct.

Ping me with one of:
- "All checks passed, take me to lesson 6 (model architecture)"
- "Check N looks off — here's what I saw"
- "Data loading is slow, want to speed it up"

Lesson 6 is the model. Then lesson 7 is the training loop. Then we have a working VAD. We're close.
